In [7]:
import os
import torch
import kagglehub
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import classification_report

In [9]:
path = kagglehub.dataset_download("serenaraju/yawn-eye-dataset-new")

print("Dataset path:", path)

Using Colab cache for faster access to the 'yawn-eye-dataset-new' dataset.
Dataset path: /kaggle/input/yawn-eye-dataset-new


In [10]:
for root, dirs, files in os.walk(path):
    print(root)

/kaggle/input/yawn-eye-dataset-new
/kaggle/input/yawn-eye-dataset-new/dataset_new
/kaggle/input/yawn-eye-dataset-new/dataset_new/test
/kaggle/input/yawn-eye-dataset-new/dataset_new/test/Closed
/kaggle/input/yawn-eye-dataset-new/dataset_new/test/Open
/kaggle/input/yawn-eye-dataset-new/dataset_new/test/yawn
/kaggle/input/yawn-eye-dataset-new/dataset_new/test/no_yawn
/kaggle/input/yawn-eye-dataset-new/dataset_new/train
/kaggle/input/yawn-eye-dataset-new/dataset_new/train/Closed
/kaggle/input/yawn-eye-dataset-new/dataset_new/train/Open
/kaggle/input/yawn-eye-dataset-new/dataset_new/train/yawn
/kaggle/input/yawn-eye-dataset-new/dataset_new/train/no_yawn


In [12]:
data_dir = path

In [13]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

Classes: ['dataset_new']
Total images: 2900


In [14]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [15]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [16]:
model = models.alexnet(pretrained=True)
model.classifier[6] = nn.Linear(4096, 2)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 161MB/s]


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [19]:
epochs = 5
best_accuracy = 0

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "drowsiness_model.pth")

    print(f"Epoch {epoch+1}")
    print(f"Loss: {running_loss/len(train_loader):.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-"*30)

Epoch 1
Loss: 0.0083
Accuracy: 100.00%
------------------------------
Epoch 2
Loss: 0.0000
Accuracy: 100.00%
------------------------------
Epoch 3
Loss: 0.0000
Accuracy: 100.00%
------------------------------
Epoch 4
Loss: 0.0000
Accuracy: 100.00%
------------------------------
Epoch 5
Loss: 0.0000
Accuracy: 100.00%
------------------------------


In [20]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred, target_names=dataset.classes))

              precision    recall  f1-score   support

 dataset_new       1.00      1.00      1.00       580

    accuracy                           1.00       580
   macro avg       1.00      1.00      1.00       580
weighted avg       1.00      1.00      1.00       580

